# Identifying cells affected by bright stars

When running the shear estimation algorithm `metadetection`, the bright stars need to be masked (and apodized), ideally at the pixel level, to not adversely affect the shear measurement.
This is because it performs Fourier transforms internally, which is non-local.
The masks are circular centered around bright stars, with their magnitudes determining the radii.
However, an appropriate magnitude-to-radius mapping was not determined ahead of running `metadetection`.
Because this relation depends on various image processing algorithms, especially background subtraction, it is not appropriate to apply the relation from previous surveys (DES Y6) to DP2.

No bright star masks were applied to produce the `ShearObject` catalog in DP2.
Instead, once a user has determined such a relation, we demonstrate how to mitigate the impact of bright stars in downstream analyses.
This involves discarding shear measurements from entire cells altogether that would have been affected by bright stars.
The idea is to identify cells that overlap any of the circular regions around the Gaia stars.
For an example `mag_to_radius` relation, we provide two different equivalent methods to identify the overlapping cells.

### Caveats

1. This notebook does not demonstrate *how* to identify a magnitude-to-radius relation
2. This works for (very) bright stars only, whose mask radii would be large enough to wipe out several cells. The marginal loss in footprint area is minimal in these cases.
3. This assumes that discarding measurements close to the faint(er) stars is sufficient and does not need pixel-level masking.

In [ ]:
from itertools import combinations, product

import lsst.geom
import matplotlib.pyplot as plt
import numpy as np
from lsst.daf.butler import Butler
from lsst.meas.algorithms import LoadReferenceObjectsConfig, ReferenceObjectLoader
from lsst.skymap import Index2D, makeSkyPolygonFromBBox
from lsst.sphgeom import Angle, Circle, HtmPixelization, LonLat, UnionRegion, UnitVector3d
from matplotlib.patches import Ellipse, Polygon

We consider a simple piecewise constant function for the magnitude-radius relation.
This is a slightly simplified example from Xiangchong Li.
This example is particularly handy as it has a definite maximum radius.
The appropriate relation for DP2 is yet to be defined.

In [ ]:
def mag_to_radius(mag):
    """Convert GAIA g-band magnitude to radius (in radians).
    """
    mag = np.asarray(mag, dtype=np.float64)
    return np.select(
        [mag <= 11.0, mag <= 14.0],
        np.array([450, 200.0]) * 0.2 * np.pi / (180 * 60 * 60),  # 450 & 200 px
        default=0.0,
    )

# Visualize it immediately
mags = np.linspace(0, 25, 251)
radii = mag_to_radius(mags) * 180 * 60 / np.pi
plt.plot(mags, radii)
plt.xlabel("Magnitude")
plt.ylabel("Radii [arcmin]")

In [ ]:
ref_loader_config = LoadReferenceObjectsConfig()

# pixelMargin has to be at least the size of the maximum pixel_radius.
# This is where a well-defined maximum radius comes in handy.
ref_loader_config.pixelMargin = 450  # arbitrary for demo purposes

# AB magnitude from nJy flux: mag = -2.5*log10(flux_nJy) + 31.4
AB_ZP_NJY = 31.4

In [ ]:
butler = Butler("dp2", collections=["dp2"])
skyMap = butler.get("skyMap", skymap="lsst_cells_v2")

To analyze a different tract, it is sufficient to change the `tractId` below and rerun the cells below.

In [ ]:
tractId = 9813  # COSMOS field

The coadded images, and the catalogs derived from it, have `(tract, patch)` dimensions, but the reference catalogs are sharded in HTM7 pixels.
But no worries, the butler is capable of doing the spatial queries for us.
But we cannot limit ourselves to the one tract we have, since there may be bright stars outside the tract, whose bright wings will still affect our tract and hence need to be masked.
To get those, we need to obtain the adjacent `tractId` s first and include them in our query.
Query for whole adjacent tracts is definitely an overkill, but we will quickly discard the non-overlapping HTM7 regions.

In [ ]:
tractInfo = skyMap[tractId]
wcs = tractInfo.getWcs()

# Bounding box for the tract, expanded by pixelMargin on all sides.
bbox = lsst.geom.Box2D(tractInfo.getBBox())
bbox.grow(ref_loader_config.pixelMargin)

tracts = []

# Convert each corner (pixel coords) to sky, then find the tract there.
corners = bbox.getCorners()
for corner in corners:
    coord = wcs.pixelToSky(corner)
    tract = skyMap.findTract(coord)
    tracts.append(tract.getId())

for corner1, corner2 in combinations(corners, 2):
    eval_point = lsst.geom.Point2D(
        0.5 * (corner1.x + corner2.x),
        0.5 * (corner1.y + corner2.y),
    )
    coord = wcs.pixelToSky(eval_point)
    tract = skyMap.findTract(coord)
    tracts.append(tract.getId())
    
print(sorted(tracts))
# Repetition of tracts is good.
tracts = np.unique(tracts)  # Unique and sorted in ascending order.

Some quick checks to see if we got all the adjacent tracts.

Because of the rings tesselation, we expect `tractId ± 1` to be in `tracts`. We also expect three different sequence of successive numbers.

In [ ]:
try:
    assert tractId - 1 in tracts, "tractId - 1 is missing"
    assert tractId + 1 in tracts, "tractId + 1 is missing"
    assert tractId in tracts, "tractId is missing"
    assert ((tracts[1:] - tracts[:-1]) > 2).sum() == 2, "Midpoint evaluation is missing some tracts"
    assert len(tracts) >= 6, "Too few tracts to be possible"
except AssertionError as e:
    print("Refer to https://github.com/lsst/skymap/blob/main/examples/plotSkyMap.py to visualize the skymap")
    raise e

In [ ]:
where = f"skymap='lsst_cells_v2' AND tract IN {tuple(set(int(t) for t in tracts))}"
print(f"Querying the butler for the monster catalog with {where=}")
ref_cat_refs = butler.query_datasets("the_monster_20250219", where=where)
print("Number of reference catalogs (htm7 regions) considered = ", len(ref_cat_refs))

### Method 1 (ReferenceObjectLoader)

This is the preferred method and is the closest to how the reference catalog will be loaded in production. This is efficient because it filters out the regions that have no overlap and then load the catalogs. While neither is time consuming, this has lesser demand on the underlying Postgres database.

But first, time for some duck typing. ReferenceObjectLoader.loadRegion accesses `dataId.region` which is typically filled from the *QuantumGraph* generated during the processing.
Since we do not have the QuantumGraph as a released product (why would we?), we have to fill in the region ourselves by the corresponding HTM7 pixel.

In [ ]:
class MyDataCoordinate:
    """Duck tying for `~lsst.daf.butler.DataCoordinate`
    """
    _htm = HtmPixelization(7)

    def __init__(self, ref):
        self._htm7 = ref.dataId["htm7"]

    @property
    def region(self):
        return self._htm.pixel(self._htm7)


# One MyDataCoordinate per ref, in the same order as refCats so the loader's
# zip(self.dataIds, self.refCats) pairs each ref with its correct region.
dataIds = [MyDataCoordinate(ref) for ref in ref_cat_refs]

# loadRegion calls refCat.get(), so refCats must be *deferred* dataset handles,
# not bare DatasetRefs (which have no .get()).
refCats = [butler.getDeferred(ref) for ref in ref_cat_refs]

ref_loader = ReferenceObjectLoader(
    dataIds=dataIds,
    refCats=refCats,
    name="the_monster_20250219",
    config=ref_loader_config,
)

In [ ]:
# Load reference objects over the tract, expanded by pixelMargin.
# There is no reason to expand it here; that is done internally.
# loadPixelBox(bbox, wcs, filterName, ...) converts the pixel box to a sphere
# region internally and pulls every shard (via dataId.region) that overlaps.
loadBBox = lsst.geom.Box2I(tractInfo.getBBox())

result = ref_loader.loadPixelBox(
    bbox=loadBBox,
    wcs=wcs,
    filterName="phot_g_mean",   # -> phot_g_mean_flux. _flux suffix is appended internally.
)
refCat = result.refCat

# Uncomment to see the catalog
# refCat.asAstropy()

In [ ]:
# Plot the magnitude distribution of the reference catalog
mags = -2.5 * np.log10(refCat["phot_g_mean_flux"]) + AB_ZP_NJY
plt.hist(mags, bins=50, density=True, histtype="step")
plt.yscale("log")
plt.xlabel("Gaia G-band magnitude")
plt.ylabel("Normalized counts")
# Plot the magnitudes where the radius transition happens.
# This is unique to our piecewise constant magnitude-radius relation.
plt.axvline(11.0, ls=':')
plt.axvline(14.0, ls=':')

Now we create circles in the sky around each bright star using our `mag_to_radius` relation.
We will calculate which cells these circles overlap and report them.

In [ ]:
# Radius per refCat row, from its phot_g_mean_flux -> AB mag -> mag_to_radius.
ra = np.asarray(refCat["coord_ra"])    # radians
dec = np.asarray(refCat["coord_dec"])  # radians
flux = np.asarray(refCat[result.fluxField])  # nJy

good = np.isfinite(flux) & (flux > 0)
mag = np.full(len(refCat), np.inf)
mag[good] = -2.5 * np.log10(flux[good]) + AB_ZP_NJY

radius = mag_to_radius(mag)  # opening angle in radians

union = UnionRegion(*(
    Circle(UnitVector3d(LonLat.fromRadians(ra[i], dec[i])), Angle(radius[i]))
    for i in range(len(radius))
    if radius[i] > 0
))
print("Number of circles (radius > 0):", len(union.getRegions(union)))

Let us first plot these visually on top of the various bounding boxes (tract, patch, cell) to get a sense for it.

In [ ]:
def bbox_to_radec(bbox):
    """Corners of a pixel Box2I/Box2D as an (N,2) array of (RA, Dec) in degrees."""
    return np.array(
        [[c.getLongitude().asDegrees(), c.getLatitude().asDegrees()]
         for c in (wcs.pixelToSky(p) for p in lsst.geom.Box2D(bbox).getCorners())]
    )
    
def plot_bright_star_masks(tractInfo, union):
    poly_radec = bbox_to_radec(tractInfo.getBBox())
    
    fig, ax = plt.subplots(figsize=(9, 9))
    
    # Patch and cell boundaries, lighter lines underneath.
    for patchInfo in tractInfo:
        ax.add_patch(Polygon(bbox_to_radec(patchInfo.outer_bbox), closed=True,
                             fill=False, edgecolor="0.6", lw=0.4, zorder=1))
        nc = patchInfo.getNumCells()
        for i in range(nc.x * nc.y):
            cbbox = patchInfo.getCellInfo(i).outer_bbox
            ax.add_patch(Polygon(bbox_to_radec(cbbox), closed=True,
                                 fill=False, edgecolor="0.85", lw=0.2, zorder=0))
    
    # Tract boundary.
    ax.add_patch(Polygon(poly_radec, closed=True, fill=False,
                         edgecolor="tab:blue", lw=2, label="tract bbox", zorder=3))
    
    # One ellipse per refCat circle; width scaled by 1/cos(dec) to look round.
    circles = union.getRegions(union)
    for circ in circles:
        center = LonLat(circ.getCenter())
        ra_c = center.getLon().asDegrees()
        dec_c = center.getLat().asDegrees()
        rad_deg = circ.getOpeningAngle().asDegrees()
        ax.add_patch(Ellipse((ra_c, dec_c),
                             width=2 * rad_deg / np.cos(np.radians(dec_c)),
                             height=2 * rad_deg,
                             fill=False, edgecolor="tab:red", lw=0.5, alpha=0.6, zorder=2))
    
    ra_min, ra_max = poly_radec[:, 0].min(), poly_radec[:, 0].max()
    dec_min, dec_max = poly_radec[:, 1].min(), poly_radec[:, 1].max()
    pad_ra = 0.1 * (ra_max - ra_min)
    pad_dec = 0.1 * (dec_max - dec_min)
    ax.set_xlim(ra_max + pad_ra, ra_min - pad_ra)  # RA increases to the left
    ax.set_ylim(dec_min - pad_dec, dec_max + pad_dec)
    
    ax.set_xlabel("RA [deg]")
    ax.set_ylabel("Dec [deg]")
    ax.set_title(f"Tract {tractId}: bbox + {len(circles)} mask circles")
    ax.set_aspect(1.0 / np.cos(np.radians(poly_radec[:, 1].mean())))
    plt.tight_layout()
    plt.show()

    return fig, ax

In [ ]:
# This is usually the time-consuming step of the whole notebook.
# When running in loop for multiple tractId values, skip this step.
plot_bright_star_masks(tractInfo, union)

In [ ]:
def iter_overlapping_cells(tractInfo, region):
    """Yield (patchId, cellId, overlap) for every cell whose outer sky polygon
    touches `region`. `overlap` is "full" if the cell lies entirely within the
    region, else "partial". patchId/cellId are sequential integer indices.

    Partial overlaps are reported (yielded with overlap="partial").
    """
    for patchInfo in tractInfo:
        # Prune whole patches that don't overlap.
        # We are being hopeful here.
        if patchInfo.inner_sky_polygon.isDisjointFrom(region):
            continue
        patchId = patchInfo.index
        # We loop over only the inner cells.
        for cx, cy in product(range(1, 22), range(1, 22)):
            cellId = Index2D(cx, cy)
            cellInfo = patchInfo.getCellInfo(cellId)
            poly = makeSkyPolygonFromBBox(cellInfo.inner_bbox.dilatedBy(50), cellInfo.wcs)
            if poly.isDisjointFrom(region):
                continue
            overlap = "full" if poly.isWithin(region) else "partial"
            yield patchId, cellId, overlap

In [ ]:
discard_cells_iterator = iter_overlapping_cells(tractInfo, union)

In [ ]:
for num, row in enumerate(discard_cells_iterator):
    print(row)
    if num > 20:
        break

### Method 2

If we don't care about the underlying database at all, just fetch all the reference catalogs from the 8+ tracts and then filter them.
This is less verbose but is less efficient.

In [ ]:
def get_all_circles():
    for ref in ref_cat_refs:
        cat = butler.get(ref)
        ra = np.asarray(cat["coord_ra"])      # radians
        dec = np.asarray(cat["coord_dec"])    # radians
        flux = np.asarray(cat["phot_g_mean_flux"])  # nJy
    
        # Only positive, finite fluxes yield a valid magnitude.
        good = np.isfinite(flux) & (flux > 0)
        mag = np.full(len(cat), np.inf)
        mag[good] = -2.5 * np.log10(flux[good]) + AB_ZP_NJY
    
        radius = mag_to_radius(mag)  # opening angle in radians
    
        for i in np.where(radius > 0)[0]:
            center = UnitVector3d(LonLat.fromRadians(ra[i], dec[i]))
            yield Circle(center, Angle(radius[i]))

union_without_refobj = UnionRegion(*get_all_circles())
print("Number of circles (radius > 0):", len(UnionRegion.getRegions(union_without_refobj)))

In [ ]:
# This is even more time-consuming that the previous call to plot_bright_star_masks as it has more patches to plot.
# When running in loop for multiple tractId values, skip this step.
plot_bright_star_masks(tractInfo, union_without_refobj)

In [ ]:
assert list(iter_overlapping_cells(tractInfo, union)) == list(iter_overlapping_cells(tractInfo, union_without_refobj))